# All-cause mortality from study entry in SUPPORT2

Run all cells offline with the default synthetic data. For the full cohort, download the
[HBiostat SUPPORT2 archive](https://hbiostat.org/data/repo/support2csv.zip) and set
`archive` below. Set `dataset_root` to load a directory containing a raw or prepared
`support2.csv`. Run this notebook from the repository's `examples` directory
with this branch of PyHealth, Matplotlib, and ipywidgets installed.

The public `Support2TimeToEvent` task uses `d.time` for observed follow-up in
days and `death` for death during follow-up. Age and sex are the study-entry
predictors. Most physiologic assessments were made on study day 3.
[SUPPORT2 field dictionary](https://archive.ics.uci.edu/dataset/880/support2) ·
[SUPPORT study notes](https://hbiostat.org/data/repo/supportdesc)

In [ ]:
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from survival_mdn_support2 import (
    CSV_NAME,
    EXPORT_DAYS,
    RELEASE_LABELS,
    ModelTimeLoader,
    export_survival,
    prepare_support2,
    write_synthetic_support2,
)

from pyhealth.datasets import PatientSplit, Support2Dataset, get_dataloader
from pyhealth.models import SurvivalMDN
from pyhealth.tasks import Support2TimeToEvent
from pyhealth.trainer import Trainer

dataset_root = None  # Or Path("/path/to/directory/containing/support2.csv")
archive = None  # Or Path("/path/to/support2csv.zip")
seed = 19
epochs = 20
output_dir = Path("output/support2-native")

torch.manual_seed(seed)
torch.set_num_threads(1)

## Prepare source records

The HBiostat ZIP has 47 named columns and 48 fields on each data row. The unnamed
first field is the row ID. The example validates the archive hash, label mapping,
and follow-up range, then writes a separate CSV with the `sno` header added.
A prepared CSV is validated and read in place. The synthetic data uses
exponential event times with a 90-day median and independent right censoring.

In [ ]:
if archive is not None and dataset_root is not None:
    raise ValueError("Choose either archive or dataset_root.")

if archive is not None:
    source = archive
elif dataset_root is not None:
    source = dataset_root / CSV_NAME
else:
    source = output_dir / "synthetic" / CSV_NAME
    write_synthetic_support2(source, seed=seed, n_patients=480)

table, source_ids = prepare_support2(source, output_dir / "prepared")
dataset = Support2Dataset(
    root=str(table.parent), tables=["support2"], cache_dir=output_dir / "cache"
)
task = Support2TimeToEvent()
if archive is not None:
    for patient_id, expected in RELEASE_LABELS.items():
        (sample,) = task(dataset.get_patient(patient_id))
        assert (sample["duration"], sample["event"]) == expected
    print("Verified native label mapping for the first three source rows.")

assert set(dataset.unique_patient_ids) == source_ids

## Split by patient and fit preprocessing

`set_task` fits age imputation and scaling and the sex vocabulary on the training
partition. The fitted processor transforms all three partitions using the same
columns and statistics. Missing age and missing or unseen sex have dedicated indicators.

In [ ]:
split = PatientSplit((0.70, 0.15, 0.15), seed=seed)
train, validation, test = dataset.set_task(task, split=split)
parts = {"train": train, "validation": validation, "test": test}
part_ids = {name: set(part.patient_to_index) for name, part in parts.items()}
if not all(part_ids.values()):
    raise ValueError("Train, validation, and test must each contain patients.")
assert all(a.isdisjoint(b) for a, b in combinations(part_ids.values(), 2))
eligible_ids = set.union(*part_ids.values())
assert eligible_ids <= source_ids
excluded = len(source_ids - eligible_ids)
assert len(source_ids) == excluded + sum(len(part) for part in parts.values())

print(f"Seed {seed} | source {len(source_ids):,} | excluded {excluded:,}")
for name, part in parts.items():
    events = sum(
        int(batch["event"].sum().item())
        for batch in get_dataloader(part, batch_size=128)
    )
    print(
        f"{name}: {len(part):,} patients, {events:,} events, {len(part) - events:,} censored"
    )

processor = train.input_processors["features"]
print(
    f"Training age median {processor.age_median}, mean {processor.age_mean:.4f}, "
    f"population SD scale {processor.age_scale:.4f}"
)
print(
    f"Training sex categories {list(processor.sex_vocabulary)} | {processor.size()} features"
)

## Select a checkpoint using validation NLL

This example trains SurvivalMDN in years. The loader adapter divides training
and validation durations by 365 as they enter the model. The task labels stay
in days. Each evaluation records full-precision validation NLL in model time.

In [ ]:
class ValidationTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.validation_losses = []

    def evaluate(self, dataloader):
        scores = super().evaluate(dataloader)
        self.validation_losses.append(scores["loss"])
        return scores


train_loader = ModelTimeLoader(get_dataloader(train, 64, shuffle=True, seed=seed))
validation_loader = ModelTimeLoader(get_dataloader(validation, 128))
model = SurvivalMDN(train, hidden_dims=(32, 32), n_components=3)
trainer = ValidationTrainer(
    model=model, device="cpu", output_path=str(output_dir), exp_name="checkpoints"
)
trainer.train(
    train_loader,
    val_dataloader=validation_loader,
    epochs=epochs,
    optimizer_params={"lr": 0.003},
    monitor="loss",
    monitor_criterion="min",
    load_best_model_at_last=True,
)
validation_losses = tuple(trainer.validation_losses)
assert len(validation_losses) == epochs and np.isfinite(validation_losses).all()
restored_nll = trainer.evaluate(validation_loader)["loss"]
np.testing.assert_allclose(restored_nll, min(validation_losses), rtol=1e-6, atol=1e-7)
print(f"Best validation epoch {np.argmin(validation_losses) + 1} of {epochs}")
print(f"Restored validation NLL {restored_nll:.8f}")

## Export held-out survival at day coordinates

Inference divides the day grid by 365 before passing it to the fitted model.
The export contains one row per test patient with estimates at 0, 30, 90, 180
and 365 days. The CSV contains patient-level predictions and identifiers.

In [ ]:
export_path = output_dir / "test_survival.csv"
ids, curves = export_survival(model, get_dataloader(test, 128), export_path)
assert len(ids) == len(set(ids)) == len(test) and set(ids) == part_ids["test"]
assert curves.shape == (len(test), len(EXPORT_DAYS))
assert np.isfinite(curves).all() and ((0 <= curves) & (curves <= 1)).all()
assert (curves[:, 0] == 1).all() and (np.diff(curves, axis=1) <= 1e-6).all()
print(
    f"Exported {curves.shape[0]:,} patients × {curves.shape[1]} horizons to {export_path}"
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(EXPORT_DAYS, curves[:12].T, color="tab:blue", alpha=0.18)
ax.plot(EXPORT_DAYS, curves.mean(axis=0), "o-", color="tab:blue", label="Test mean")
ax.set(xlabel="Days from study entry", ylabel="Predicted survival", ylim=(0, 1.03))
ax.set_xticks(EXPORT_DAYS)
ax.legend()
plt.show()